# Language-DIM Safety Analysis

This notebook analyzes safety evaluation outputs in:

- `results/language_dim/.../generations/eval/*.json`

It builds summary tables:

- baseline unsafe-rate
- per-language and per-coefficient unsafe-rate
- delta vs baseline
- pivot table (`language x coeff`)

and saves CSV files into the `summary/` folder.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Change this path if you analyze a different run
EVAL_DIR = Path("./results/language_dim/Llama-3.1-8B-Instruct_layer16/last_user_token/generations/eval")
SUMMARY_DIR = EVAL_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted([p for p in EVAL_DIR.glob("*.json") if p.name != "summary.json"])
print(f"Eval dir: {EVAL_DIR}")
print(f"Found {len(json_files)} evaluated files")

In [ ]:
def normalize_judgment(obj):
    if isinstance(obj, dict):
        return obj.get("judgment")
    return None

baseline_rows = []
steered_rows = []

for fp in json_files:
    with fp.open("r", encoding="utf-8") as f:
        data = json.load(f)

    prompt = data.get("prompt", "")
    token_mode = data.get("token_mode", "unknown")
    baseline_judgment = data.get("baseline_evaluation", {}).get("judgment")

    baseline_rows.append({
        "file": fp.name,
        "prompt": prompt,
        "token_mode": token_mode,
        "judgment": baseline_judgment,
    })

    for lang, coeff_map in data.get("steered_responses", {}).items():
        for coeff, eval_obj in coeff_map.items():
            steered_rows.append({
                "file": fp.name,
                "prompt": prompt,
                "token_mode": token_mode,
                "language": lang,
                "coeff": float(coeff),
                "judgment": normalize_judgment(eval_obj),
            })

baseline_df = pd.DataFrame(baseline_rows)
steered_df = pd.DataFrame(steered_rows)

print("baseline rows:", len(baseline_df))
print("steered rows:", len(steered_df))

baseline_df.head(3)

In [ ]:
baseline_safe = (baseline_df["judgment"] == "safe").sum()
baseline_unsafe = (baseline_df["judgment"] == "unsafe").sum()
baseline_total = baseline_safe + baseline_unsafe
baseline_unsafe_rate = (baseline_unsafe / baseline_total) if baseline_total else 0.0

print(f"Baseline unsafe rate: {baseline_unsafe_rate:.4f} ({baseline_unsafe}/{baseline_total})")

agg = (
    steered_df[steered_df["judgment"].isin(["safe", "unsafe"])]
    .groupby(["token_mode", "language", "coeff"], as_index=False)
    .agg(
        safe=("judgment", lambda s: (s == "safe").sum()),
        unsafe=("judgment", lambda s: (s == "unsafe").sum()),
    )
)
agg["n"] = agg["safe"] + agg["unsafe"]
agg["unsafe_rate"] = agg["unsafe"] / agg["n"]
agg["delta_vs_baseline"] = agg["unsafe_rate"] - baseline_unsafe_rate

summary_table = agg.sort_values(["delta_vs_baseline", "unsafe_rate"], ascending=False)
summary_table.head(30)

In [ ]:
pivot = summary_table.pivot_table(
    index=["token_mode", "language"],
    columns="coeff",
    values="unsafe_rate",
    aggfunc="mean",
)

pivot

In [ ]:
summary_csv = SUMMARY_DIR / "language_dim_summary_table.csv"
pivot_csv = SUMMARY_DIR / "language_dim_pivot_unsafe_rate.csv"
baseline_csv = SUMMARY_DIR / "language_dim_baseline_table.csv"

summary_table.to_csv(summary_csv, index=False)
pivot.reset_index().to_csv(pivot_csv, index=False)
baseline_df.to_csv(baseline_csv, index=False)

print("Saved:")
print(" -", summary_csv)
print(" -", pivot_csv)
print(" -", baseline_csv)

summary_table.head(10)

## JailbreakBench Category Table

Rows: JailbreakBench `Category` + `Overall`.

Columns: steering coefficient.

Values: unsafe rate averaged across all languages.

In [ ]:
import re
from datasets import load_dataset

# Build prompt_index -> category mapping from JailbreakBench harmful split
jbb = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")
idx_to_category = {int(row["Index"]): row["Category"] for row in jbb}

# Parse prompt index from file names like: prompt_53_...
steered_with_idx = steered_df.copy()
steered_with_idx["prompt_index"] = steered_with_idx["file"].str.extract(r"^prompt_(\d+)_")[0].astype(float)
steered_with_idx = steered_with_idx.dropna(subset=["prompt_index"])
steered_with_idx["prompt_index"] = steered_with_idx["prompt_index"].astype(int)
steered_with_idx["category"] = steered_with_idx["prompt_index"].map(idx_to_category).fillna("Unknown")

valid = steered_with_idx[steered_with_idx["judgment"].isin(["safe", "unsafe"])].copy()
valid["is_unsafe"] = (valid["judgment"] == "unsafe").astype(int)

valid.head(3)

In [ ]:
# Average across all languages: one unsafe-rate per (category, coeff)
category_coeff = (
    valid.groupby(["category", "coeff"], as_index=False)
    .agg(unsafe_rate=("is_unsafe", "mean"), n=("is_unsafe", "size"))
)

category_pivot = category_coeff.pivot(index="category", columns="coeff", values="unsafe_rate")

# Add Overall row (across all categories and languages)
overall = (
    valid.groupby("coeff", as_index=False)
    .agg(unsafe_rate=("is_unsafe", "mean"), n=("is_unsafe", "size"))
)
overall_row = pd.DataFrame([{"category": "Overall", **{row.coeff: row.unsafe_rate for _, row in overall.iterrows()}}])

category_table = category_pivot.reset_index()
category_table = pd.concat([category_table, overall_row], ignore_index=True)

# Optional: sort categories alphabetically, keep Overall last
category_table_no_overall = category_table[category_table["category"] != "Overall"].sort_values("category")
category_table_overall = category_table[category_table["category"] == "Overall"]
category_table_final = pd.concat([category_table_no_overall, category_table_overall], ignore_index=True)

# Pretty print as percentages
coeff_cols = [c for c in category_table_final.columns if c != "category"]
category_table_display = category_table_final.copy()
for c in coeff_cols:
    category_table_display[c] = category_table_display[c].map(lambda x: f"{x*100:.2f}%" if pd.notna(x) else "")

category_table_display

## Overall Language Unsafe Ranking

Sort languages by overall unsafe rate (aggregated across all prompts and coefficients).

In [ ]:
# Exclude English branch for language_dim safety ranking (as in evaluation setup)
lang_valid = valid.copy()
lang_valid = lang_valid[lang_valid["language"].str.lower() != "en"]

language_overall = (
    lang_valid.groupby("language", as_index=False)
    .agg(
        unsafe_rate=("is_unsafe", "mean"),
        n=("is_unsafe", "size"),
    )
    .sort_values("unsafe_rate", ascending=False)
    .reset_index(drop=True)
)

language_overall_display = language_overall.copy()
language_overall_display["unsafe_rate"] = language_overall_display["unsafe_rate"].map(lambda x: f"{x*100:.2f}%")

language_overall_display

In [ ]:
language_overall_csv = SUMMARY_DIR / "language_dim_language_overall_ranking.csv"
language_overall.to_csv(language_overall_csv, index=False)
print("Saved:", language_overall_csv)

language_overall_display.head(20)

## Steering Vector Norms

Compute L2 norms for:

- each language representation vector `||r_lang||`
- each effective steering vector `||r_target - r_source||` (source defaults to `en`)

In [ ]:
import torch

# Adjust path if needed (token_mode-specific vectors)
# VECTOR_PATH = Path("./results/language_dim/Llama-3.1-8B-Instruct_layer16/last_user_token/vectors/language_vectors.pt")
VECTOR_PATH = Path("./results/language_dim/Qwen2.5-7B-Instruct_layer14/last_user_token/vectors/language_vectors.pt")
SOURCE_LANG = "en"

vecs = torch.load(VECTOR_PATH, map_location="cpu")

# 1) Norm per language vector
vec_norm_rows = []
for lang, vec in vecs.items():
    vec_norm_rows.append({
        "language": lang,
        "vector_norm": float(vec.float().norm().item()),
    })

vec_norm_df = pd.DataFrame(vec_norm_rows).sort_values("vector_norm", ascending=False).reset_index(drop=True)
vec_norm_df

In [ ]:
# 2) Norm of effective steering vectors: ||r_target - r_source||
if SOURCE_LANG not in vecs:
    raise ValueError(f"SOURCE_LANG '{SOURCE_LANG}' not found in vectors")

src_vec = vecs[SOURCE_LANG].float()
eff_norm_rows = []
for target_lang, target_vec in vecs.items():
    eff_vec = target_vec.float() - src_vec
    eff_norm_rows.append({
        "source_language": SOURCE_LANG,
        "target_language": target_lang,
        "effective_vector_norm": float(eff_vec.norm().item()),
    })

eff_norm_df = (
    pd.DataFrame(eff_norm_rows)
    .sort_values("effective_vector_norm", ascending=False)
    .reset_index(drop=True)
)

eff_norm_df